# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
%pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Program Files\Python39\python.exe -m pip install --upgrade pip' command.


In [3]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

In [22]:
spaceship_clean = spaceship.dropna().copy()

# Reduce Cabin to deck letter
spaceship_clean["Cabin"] = spaceship_clean["Cabin"].str[0]

# Remove identifiers that are too specific
spaceship_clean = spaceship_clean.drop(
    columns=["PassengerId", "Name"]
)

# Separate features and target
X= spaceship_clean.drop(columns=["Transported"])
y = spaceship_clean["Transported"].astype(int)

In [23]:
import numpy as np

numerical_columns = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_columns = X.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical columns:", numerical_columns)
print("Categorical columns:", categorical_columns)

Numerical columns: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
Categorical columns: ['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP']


In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=0,
    stratify=y
)

In [25]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            StandardScaler(),
            numerical_columns
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_columns
        )
    ]
)

**Perform Train Test Split**

In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=0,
    stratify=y
)

In [27]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [28]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier

bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(
        random_state=0
    ),
    n_estimators=100,
    bootstrap=True,
    random_state=0
)

bagging_model.fit(
    X_train_processed,
    y_train
)

y_pred_bagging = bagging_model.predict(
    X_test_processed
)

In [33]:
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import pandas as pd

In [34]:
def evaluate_classifier(model_name, model):
    
    # Fit the model
    model.fit(X_train_processed, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_processed)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    # Print results
    print(model_name)
    print("Accuracy:", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F1 Score:", round(f1, 4))
    
    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    }

In [35]:
bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(
        random_state=0
    ),
    n_estimators=100,
    bootstrap=True,
    random_state=0
)

bagging_results = evaluate_classifier(
    "Bagging",
    bagging_model
)

Bagging
Accuracy: 0.798
Precision: 0.8083
Recall: 0.7853
F1 Score: 0.7966


In [36]:
pasting_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(
        random_state=0
    ),
    n_estimators=100,
    bootstrap=False,
    random_state=0
)

pasting_results = evaluate_classifier(
    "Pasting",
    pasting_model
)

Pasting
Accuracy: 0.736
Precision: 0.7268
Recall: 0.7628
F1 Score: 0.7443


- Random Forests

In [29]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=0
)

random_forest_model.fit(
    X_train_processed,
    y_train
)

y_pred_random_forest = random_forest_model.predict(
    X_test_processed
)

In [37]:
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=0,
    n_jobs=-1
)

random_forest_results = evaluate_classifier(
    "Random Forest",
    random_forest_model
)

Random Forest
Accuracy: 0.7882
Precision: 0.8016
Recall: 0.7703
F1 Score: 0.7856


- Gradient Boosting

In [30]:
from sklearn.ensemble import GradientBoostingClassifier

gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=0
)

gradient_boosting_model.fit(
    X_train_processed,
    y_train
)

y_pred_gradient = gradient_boosting_model.predict(
    X_test_processed
)

In [38]:
gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=0
)

gradient_boosting_results = evaluate_classifier(
    "Gradient Boosting",
    gradient_boosting_model
)

Gradient Boosting
Accuracy: 0.8041
Precision: 0.7799
Recall: 0.8514
F1 Score: 0.8141


- Adaptive Boosting

In [31]:
from sklearn.ensemble import AdaBoostClassifier

ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(
        max_depth=3,
        random_state=0
    ),
    n_estimators=100,
    learning_rate=0.1,
    random_state=0
)

ada_model.fit(
    X_train_processed,
    y_train
)

y_pred_ada = ada_model.predict(
    X_test_processed
)

In [39]:
ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(
        max_depth=3,
        random_state=0
    ),
    n_estimators=100,
    learning_rate=0.1,
    random_state=0
)

ada_results = evaluate_classifier(
    "AdaBoost",
    ada_model
)

AdaBoost
Accuracy: 0.8018
Precision: 0.7877
Recall: 0.8303
F1 Score: 0.8085


In [40]:
results = [
    bagging_results,
    pasting_results,
    random_forest_results,
    gradient_boosting_results,
    ada_results
]

results_df = pd.DataFrame(results)

results_df.sort_values(
    by="F1 Score",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score
3,Gradient Boosting,0.804085,0.779917,0.851351,0.814070
4,AdaBoost,0.801815,0.787749,0.830330,0.808480
0,Bagging,0.798033,0.808346,0.785285,0.796649
2,Random Forest,0.788200,0.801562,0.770270,0.785605
1,Pasting,0.736006,0.726753,0.762763,0.744322


Which model is the best and why?

### Best Model

Gradient Boosting performed best overall. It achieved the highest accuracy (**0.8041**), recall (**0.8514**), and F1 score (**0.8141**) of all the models tested.

The high recall means it correctly identified the largest proportion of passengers who were transported, while the highest F1 score shows that it provided the best overall balance between precision and recall. Although Bagging had slightly higher precision, Gradient Boosting produced the strongest overall classification performance.